# Autoregressive Decoding and Evaluation Lab

A standalone notebook for experimenting with **decoding strategies** and **language-model evaluation**. It can use a trained MiniGPT checkpoint from Notebook 04, or it runs independently with a small built-in corpus and a compact character-level model.

You will compare:
- Greedy decoding
- Temperature sampling
- Top-k sampling
- Top-p (nucleus) sampling
- Repetition penalties
- Cross-entropy and character-level perplexity
- How decoding changes output diversity (decoding changes generation, not the model's learned probabilities)

**Important:** this is an educational lab. The built-in corpus/model is only for verifying the pipeline; use a larger corpus and a trained checkpoint for meaningful experiments.


## 1. Imports and configuration

If you have `04_mini_gpt_from_scratch.pt` from Notebook 04 in this directory, set `LOAD_CHECKPOINT = True`. Otherwise, this notebook trains a small model itself so it remains standalone.


In [1]:

from pathlib import Path
import math, random, json, copy
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

LOAD_CHECKPOINT = False  # Set True if the Notebook 04 checkpoint is available.
CHECKPOINT_PATH = Path("04_mini_gpt_from_scratch.pt")

# Small defaults make this a quick CPU-capable exercise.
BLOCK_SIZE = 64
BATCH_SIZE = 16
EMBED_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 1
DROPOUT = 0.0
LEARNING_RATE = 5e-4
EPOCHS = 500
GRAD_CLIP = 1.0


Device: cuda



## 2. Load data and construct a character vocabulary

Use your own `corpus.txt` for better experiments. This notebook keeps a chronological split so later text is held out for validation and testing.


In [2]:

CORPUS_PATH = Path("/content/corpus.txt")
DEMO_TEXT = """
Autoregressive language models predict the next character from preceding context.
Greedy decoding chooses the highest probability token at every generation step.
Temperature rescales logits and controls how concentrated the sampling distribution is.
Top-k sampling retains only the k highest scoring candidates before sampling.
Top-p sampling retains the smallest candidate set whose cumulative probability reaches p.
Repetition penalties discourage repeatedly producing the same token or phrase.
Cross entropy measures the average negative log probability assigned to the correct target.
Perplexity is the exponential of average cross entropy and lower is better on the same evaluation setup.
Decoding settings affect generated text but do not retrain or change the underlying model weights.
"""
if CORPUS_PATH.exists():
    raw_text = CORPUS_PATH.read_text(encoding="utf-8")
    corpus_source = str(CORPUS_PATH)
else:
    raw_text = DEMO_TEXT.strip() + "\n"
    corpus_source = "built-in demo corpus"
raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

if len(raw_text) < 300:
    raise ValueError("Corpus too short. Add more text to corpus.txt (at least 300 characters).")

n = len(raw_text)
train_end, val_end = int(0.8 * n), int(0.9 * n)
train_text, val_text, test_text = raw_text[:train_end], raw_text[train_end:int(0.9*n)], raw_text[int(0.9*n):]

itos = ["<unk>"] + sorted(set(train_text))
stoi = {ch: i for i, ch in enumerate(itos)}
UNK_ID = stoi["<unk>"]

def encode(text):
    return [stoi.get(ch, UNK_ID) for ch in text]

def decode(ids):
    return "".join(itos[i] if i < len(itos) and i != UNK_ID else "�" for i in ids)

train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids = torch.tensor(encode(val_text), dtype=torch.long)
test_ids = torch.tensor(encode(test_text), dtype=torch.long)

print("Corpus source:", corpus_source)
print(f"Characters={n:,}; train={len(train_ids):,}; val={len(val_ids):,}; test={len(test_ids):,}")
print("Vocabulary size:", len(itos))
if corpus_source == "built-in demo corpus":
    print("Demo data is for pipeline verification only.")


Corpus source: /content/corpus.txt
Characters=10,831; train=8,664; val=1,083; test=1,084
Vocabulary size: 114



## 3. Define the model

This is a compact GPT-like decoder-only Transformer. It includes causal attention, learned positional embeddings, residual blocks, and a next-character output head. This duplicated model definition lets the notebook run independently.


In [3]:

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(block_size, block_size, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        if hasattr(F, "scaled_dot_product_attention"):
            out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=self.causal_mask[:T, :T],
                dropout_p=self.attn_dropout.p if self.training else 0.0,
            )
        else:
            scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
            scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
            weights = self.attn_dropout(torch.softmax(scores, dim=-1))
            out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffwd = FeedForward(embed_dim, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, block_size, dropout)
            for _ in range(num_layers)
        ])
        self.final_ln = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds context window {self.block_size}.")
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)[None, :, :]
        x = self.blocks(x)
        logits = self.lm_head(self.final_ln(x))
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss



## 4. Build datasets and train or load a model

If loading a checkpoint, the vocabulary and architecture are taken from that checkpoint. The checkpoint must have been trained with the same `corpus.txt` vocabulary if you want its learned token mapping to match your text.


In [4]:

class NextCharDataset(Dataset):
    def __init__(self, ids, block_size):
        self.ids, self.block_size = ids, block_size
    def __len__(self):
        return max(0, len(self.ids) - self.block_size)
    def __getitem__(self, idx):
        return self.ids[idx:idx+self.block_size], self.ids[idx+1:idx+self.block_size+1]

def make_loader(ids, shuffle=False):
    ds = NextCharDataset(ids, BLOCK_SIZE)
    if len(ds) == 0:
        raise ValueError("A data split is shorter than BLOCK_SIZE. Use more data or reduce BLOCK_SIZE.")
    return DataLoader(ds, batch_size=min(BATCH_SIZE, len(ds)), shuffle=shuffle)

train_loader = make_loader(train_ids, shuffle=True)
val_loader = make_loader(val_ids)
test_loader = make_loader(test_ids)

def evaluate(model, loader):
    model.eval()
    total_loss, total_chars = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, loss = model(x, y)
            total_loss += loss.item() * y.numel()
            total_chars += y.numel()
    avg = total_loss / max(total_chars, 1)
    return avg, math.exp(min(avg, 20.0))

if LOAD_CHECKPOINT:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    itos = checkpoint["vocab"]
    stoi = {ch: i for i, ch in enumerate(itos)}
    UNK_ID = stoi["<unk>"]
    BLOCK_SIZE = checkpoint["block_size"]
    EMBED_DIM = checkpoint["embed_dim"]
    NUM_HEADS = checkpoint["num_heads"]
    NUM_LAYERS = checkpoint["num_layers"]
    DROPOUT = checkpoint["dropout"]
    # Re-encode splits with the checkpoint vocabulary.
    def encode(text):
        return [stoi.get(ch, UNK_ID) for ch in text]
    train_ids = torch.tensor(encode(train_text), dtype=torch.long)
    val_ids = torch.tensor(encode(val_text), dtype=torch.long)
    test_ids = torch.tensor(encode(test_text), dtype=torch.long)
    train_loader, val_loader, test_loader = make_loader(train_ids, True), make_loader(val_ids), make_loader(test_ids)
    model = MiniGPT(len(itos), BLOCK_SIZE, EMBED_DIM, NUM_HEADS, NUM_LAYERS, DROPOUT).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded checkpoint:", CHECKPOINT_PATH)
else:
    model = MiniGPT(len(itos), BLOCK_SIZE, EMBED_DIM, NUM_HEADS, NUM_LAYERS, DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    best_val_loss, best_state, history = float("inf"), None, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_chars = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            _, loss = model(x, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            total_loss += loss.item() * y.numel()
            total_chars += y.numel()
        train_loss = total_loss / max(total_chars, 1)
        val_loss, val_ppl = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_char_perplexity": val_ppl})
        if epoch == 1 or epoch%100==0 or epoch==EPOCHS:
          print(f"Epoch {epoch}/{EPOCHS} | train loss={train_loss:.4f} | val loss={val_loss:.4f} | val char-PPL={val_ppl:.2f}")
        if val_loss < best_val_loss:
            best_val_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)


Epoch 1/500 | train loss=5.8352 | val loss=2.8078 | val char-PPL=16.57
Epoch 100/500 | train loss=0.1548 | val loss=6.7352 | val char-PPL=841.54
Epoch 200/500 | train loss=0.1318 | val loss=6.8594 | val char-PPL=952.81
Epoch 300/500 | train loss=0.1236 | val loss=6.7016 | val char-PPL=813.74
Epoch 400/500 | train loss=0.1199 | val loss=6.6437 | val char-PPL=767.92
Epoch 500/500 | train loss=0.1173 | val loss=6.8353 | val char-PPL=930.15



## 5. Implement decoding strategies

These functions change how we select the next character from the model's logits:
- **Greedy:** choose the most likely character.
- **Temperature:** divide logits by a positive temperature before softmax.
- **Top-k:** keep only the `k` highest-logit candidates.
- **Top-p:** keep the smallest high-probability set whose cumulative probability reaches `p`.
- **Repetition penalty:** lower the scores of previously generated token IDs.

For a character model, repetition penalties operate on individual characters, not semantic phrases.


In [5]:

def apply_repetition_penalty(logits, generated_ids, penalty=1.0):
    logits = logits.clone()
    if penalty <= 0:
        raise ValueError("repetition penalty must be positive")
    if penalty == 1.0 or not generated_ids:
        return logits
    for token_id in set(generated_ids):
        if logits[0, token_id] < 0:
            logits[0, token_id] *= penalty
        else:
            logits[0, token_id] /= penalty
    return logits

def filter_top_k(logits, k):
    if k is None or k <= 0 or k >= logits.size(-1):
        return logits
    threshold = torch.topk(logits, k, dim=-1).values[:, -1].unsqueeze(-1)
    return logits.masked_fill(logits < threshold, float("-inf"))

def filter_top_p(logits, p):
    if p is None or p >= 1.0:
        return logits
    if p <= 0:
        raise ValueError("top_p must be in (0, 1].")
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    # Remove tokens after the first cumulative probability that reaches p.
    remove = cumulative_probs > p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    remove_original_order = torch.zeros_like(remove).scatter(1, sorted_indices, remove)
    return logits.masked_fill(remove_original_order, float("-inf"))

@torch.no_grad()
def generate(model, prompt, max_new_chars=250, strategy="top_p", temperature=0.8,
             top_k=10, top_p=0.9, repetition_penalty=1.0):
    model.eval()
    prompt_ids = encode(prompt)
    if not prompt_ids:
        prompt_ids = [stoi.get("\n", UNK_ID)]
    generated = list(prompt_ids)
    for _ in range(max_new_chars):
        context = torch.tensor([generated[-model.block_size:]], dtype=torch.long, device=DEVICE)
        logits, _ = model(context)
        next_logits = logits[:, -1, :].clone()
        next_logits = apply_repetition_penalty(next_logits, generated, repetition_penalty)

        if strategy == "greedy":
            next_id = torch.argmax(next_logits, dim=-1, keepdim=True)
        else:
            next_logits = next_logits / max(float(temperature), 1e-5)
            if strategy == "top_k":
                next_logits = filter_top_k(next_logits, top_k)
            elif strategy == "top_p":
                next_logits = filter_top_p(next_logits, top_p)
            elif strategy == "temperature":
                pass
            else:
                raise ValueError("strategy must be greedy, temperature, top_k, or top_p")
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
        generated.append(next_id.item())
    return decode(generated)



## 6. Compare generation strategies

Run the same prompt through each decoding strategy. Random sampling can vary between runs; greedy decoding is deterministic for a fixed model and prompt.


In [6]:

prompt ="Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden "
print("PROMPT:", repr(prompt))

settings = [
    {"strategy": "greedy"},
    {"strategy": "temperature", "temperature": 0.7},
    {"strategy": "top_k", "temperature": 0.8, "top_k": 5},
    {"strategy": "top_p", "temperature": 0.8, "top_p": 0.9},
    {"strategy": "top_p", "temperature": 0.8, "top_p": 0.9, "repetition_penalty": 1.2},
]

for config in settings:
    result = generate(
        model, prompt, max_new_chars=180,
        temperature=config.get("temperature", 1.0),
        top_k=config.get("top_k", 10),
        top_p=config.get("top_p", 0.9),
        repetition_penalty=config.get("repetition_penalty", 1.0),
        strategy=config["strategy"],
    )
    print("\n" + "=" * 80)
    print("CONFIG:", config)
    print(result)


PROMPT: 'Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden '

CONFIG: {'strategy': 'greedy'}
Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden tor the Naruto ->|Teammailly Sasuki  Naruto ->0               Naruto ---|-----------------------------------------------------------------------------------------------------------

CONFIG: {'strategy': 'temperature', 'temperature': 0.7}
Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden Sparand asurof the **Sarader Sousasuke Treame cs promerss), tor-msenes int cears ind chee ties loning tnin Sharuinga asurt setnd winenjaraicl buker cht Sharis evend bras cores, prc

CONFIG: {'strategy': 'top_k', 'temperature': 0.8, 'top_k': 5}
Naruto Uzumaki (うずまき ナルト) – a shinobi of Konohagakure (Hidden tathers **Severnd **Sharishim Nouto o (villey fortim the Naruto’s truent ind and acreaming s arries tin aservechipsts. Che **Naruto* Uzuyaki an thee Umbyskrai* hapher (Paicserssom 

CONFIG: {'strategy': 'top_p', 'temperature': 


## 7. Evaluate the model and compare held-out loss

Perplexity is computed from cross-entropy. It evaluates the model's probability assignments, not the subjective quality of samples. Different decoding settings do **not** change this metric because they do not change model weights or the raw model distribution.


In [7]:

test_loss, test_ppl = evaluate(model, test_loader)
val_loss, val_ppl = evaluate(model, val_loader)
print(f"Validation cross-entropy: {val_loss:.4f} | character-PPL: {val_ppl:.2f}")
print(f"Test cross-entropy:       {test_loss:.4f} | character-PPL: {test_ppl:.2f}")

# Tiny qualitative proxy: unique-character ratio and repeated adjacent character rate.
def sample_stats(sample):
    chars = sample[len(prompt):]
    if not chars:
        return {"unique_char_ratio": 0.0, "adjacent_repeat_rate": 0.0}
    unique_ratio = len(set(chars)) / len(chars)
    repeat_rate = sum(chars[i] == chars[i-1] for i in range(1, len(chars))) / max(len(chars)-1, 1)
    return {"unique_char_ratio": round(unique_ratio, 3), "adjacent_repeat_rate": round(repeat_rate, 3)}

for strategy in ["greedy", "temperature", "top_k", "top_p"]:
    sample = generate(model, prompt, max_new_chars=200, strategy=strategy, temperature=0.8, top_k=8, top_p=0.9)
    print(strategy, sample_stats(sample))


Validation cross-entropy: 2.3045 | character-PPL: 10.02
Test cross-entropy:       2.1708 | character-PPL: 8.76
greedy {'unique_char_ratio': 0.105, 'adjacent_repeat_rate': 0.729}
temperature {'unique_char_ratio': 0.18, 'adjacent_repeat_rate': 0.055}
top_k {'unique_char_ratio': 0.175, 'adjacent_repeat_rate': 0.06}
top_p {'unique_char_ratio': 0.15, 'adjacent_repeat_rate': 0.03}



## 8. Save the model and experiment configuration

This saves a checkpoint and JSON report. For a model loaded from Notebook 04, saving a new checkpoint is optional; the notebook still exports the current weights and experiment settings.


In [8]:

checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab": itos,
    "block_size": model.block_size,
    "embed_dim": EMBED_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
}
torch.save(checkpoint, "05_decoding_lab_model.pt")

report = {
    "corpus_source": corpus_source,
    "vocab_size": len(itos),
    "validation_cross_entropy": val_loss,
    "validation_char_perplexity": val_ppl,
    "test_cross_entropy": test_loss,
    "test_char_perplexity": test_ppl,
    "training_history": history if not LOAD_CHECKPOINT else [],
}
with open("05_decoding_lab_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Saved 05_decoding_lab_model.pt")
print("Saved 05_decoding_lab_report.json")


Saved 05_decoding_lab_model.pt
Saved 05_decoding_lab_report.json



## 9. Key takeaways and experiments

- Lower temperature makes sampling more concentrated; higher temperature generally increases diversity.
- Top-k caps the candidate set by count; top-p adapts the candidate set to the probability distribution.
- Greedy decoding can become repetitive because it always chooses the single highest-probability token.
- Repetition penalties can reduce loops but may also suppress useful repeated characters or words.
- Perplexity evaluates predictive fit; generation quality also needs qualitative review and task-specific evaluation.

Try multiple prompts and compare settings. On a real text corpus, use held-out samples and avoid interpreting metrics from the tiny built-in demo as meaningful model performance.
